In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import mlflow
import mlflow.sklearn

In [8]:
df = sns.load_dataset("penguins").dropna()
df[(df['species'] == 'Chinstrap') | (df['species'] == 'Gentoo')]


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
152,Chinstrap,Dream,46.5,17.9,192.0,3500.0,Female
153,Chinstrap,Dream,50.0,19.5,196.0,3900.0,Male
154,Chinstrap,Dream,51.3,19.2,193.0,3650.0,Male
155,Chinstrap,Dream,45.4,18.7,188.0,3525.0,Female
156,Chinstrap,Dream,52.7,19.8,197.0,3725.0,Male
...,...,...,...,...,...,...,...
338,Gentoo,Biscoe,47.2,13.7,214.0,4925.0,Female
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [41]:
features = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
X = df[features]
y = df['species']
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
 X, y,
 test_size=0.2,
 random_state=42, stratify=y
)

In [ ]:
from sklearn.linear_model import LogisticRegression

# X_train_transformed = preprocessor.fit_transform(X_train)
# X_test_transformed = preprocessor.transform(X_test)
params = { "max_iter": 1000, "random_state": 42}

log_reg = LogisticRegression(**params)
log_reg.fit(X_train, y_train)

In [42]:
# from sklearn.preprocessing import OneHotEncoder

# cat_col = X.select_dtypes(include=['object']).columns
# categorical_preprocessor = OneHotEncoder(handle_unknown="ignore")

# from sklearn.compose import ColumnTransformer

# preprocessor = ColumnTransformer([
#     ('one-hot-encoder', categorical_preprocessor, list(cat_col))])

In [ ]:
import joblib
from mlflow.models import infer_signature
# mlflow.set_experiment("penguin-classification")
# mlflow.set_tracking_uri("127.0.0.1:5000")

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("penguin-classification")
with mlflow.start_run():
        
        log_reg.fit(X_train, y_train)

        pred= log_reg.predict(X_test)

        accuracy = accuracy_score(y_test, pred)
        precision = precision_score(y_test, pred, average="weighted")
        recall = recall_score(y_test, pred, average="weighted")
        f1 = f1_score(y_test, pred, average="weighted")
        # Log parameters
        mlflow.log_params(log_reg.get_params())
        # Log metrics
        mlflow.log_metrics({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1})

        # Log metadata
        mlflow.set_tag("model_name", log_reg)
        mlflow.set_tag("project", "penguin-classification")

        signature = infer_signature(
            X_train,
            log_reg.predict(X_train)
        )
        mlflow.sklearn.log_model(
            sk_model=log_reg,
            name="model",
            signature=signature,
            input_example=X_train.head(5),
            registered_model_name="penguin-classification"
        )
        joblib.dump(log_reg, "log_reg.joblib")
        mlflow.log_artifact("log_reg.joblib")   
        print(f"Model: {log_reg}")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"F1 Score: {f1:.4f}")

/opt/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model: LogisticRegression()
Accuracy: 0.9701
F1 Score: 0.9700


Registered model 'penguin-log-reg' already exists. Creating a new version of this model...
Created version '3' of model 'penguin-log-reg'.


In [45]:
mlflow.sklearn.log_model(log_reg, "log_reg_model", registered_model_name="penguin-log-reg")


2026/11/25 15:41:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
Registered model 'penguin-log-reg' already exists. Creating a new version of this model...
Created version '4' of model 'penguin-log-reg'.


In [ ]:
model_uri = f"runs:/{"run_id"}/model"
registered_model = mlflow.register_model(
    model_uri=model_uri,
    name="penguin-classifier"
)
# with mlflow.start_run() as run:
#     run_id = run.info.run_id


Registered model 'penguin-classifier' already exists. Creating a new version of this model...


MlflowException: Run with id=run_id not found

In [ ]:
import joblib

filename = 'log_reg.joblib'
joblib.dump(log_reg, filename)

print(f"Model saved to {filename}")


Model saved to log_reg.joblib


In [ ]:
print(type(df))
print(f"Target names: {list(df['species'].unique())}")


<class 'pandas.core.frame.DataFrame'>
Target names: ['Adelie', 'Chinstrap', 'Gentoo']


In [ ]:
print(df.columns.tolist())
# ['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']

print(df.dtypes)   # покажет, где float64, а где object (строки)
print(df.head(1))  # покажет реальные значения для примера (example)


['species', 'island', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g', 'sex']
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object
  species     island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
0  Adelie  Torgersen            39.1           18.7              181.0   

   body_mass_g   sex  
0       3750.0  Male  
